In [7]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score
import joblib

In [8]:
data_path = "../data/heart_disease_selected.csv"
df = pd.read_csv(data_path)

X = df.drop(columns=["target"])
y = df["target"]

# ✅ Robust label handling
if set(y.unique()) <= {0, 1}:
    y = y.astype(int)
elif set(y.unique()) <= {-1, 0}:
    y = y.replace(-1, 1).astype(int)
else:
    y = (y > 0).astype(int)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
param_grid = {
    "Logistic Regression": {
        "model": LogisticRegression(solver="liblinear", max_iter=1000),
        "params": {
            "C": [0.01, 0.1, 1, 10],
            "penalty": ["l1", "l2"]
        }
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            "max_depth": [2, 4, 6, 8, None],
            "min_samples_split": [2, 5, 10]
        }
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [50, 100, 200],
            "max_depth": [4, 6, 8, None],
            "min_samples_split": [2, 5, 10]
        }
    },
    "SVM": {
        "model": SVC(probability=True, random_state=42),
        "params": {
            "C": [0.1, 1, 10],
            "kernel": ["linear", "rbf", "poly"]
        }
    }
}

In [11]:
best_models = {}
for name, config in param_grid.items():
    print(f"\n🔹 Tuning {name}...")
    grid = GridSearchCV(
        config["model"],
        config["params"],
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"Best Params for {name}: {grid.best_params_}")
    print(f"Best CV Accuracy: {grid.best_score_:.4f}")

    best_model = grid.best_estimator_
    best_models[name] = best_model

    # Evaluate on test set
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, "predict_proba") else None
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    if y_proba is not None:
        print("ROC-AUC:", roc_auc_score(y_test, y_proba))



🔹 Tuning Logistic Regression...
Best Params for Logistic Regression: {'C': 0.01, 'penalty': 'l2'}
Best CV Accuracy: 0.8297

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.82      0.82        33
           1       0.79      0.79      0.79        28

    accuracy                           0.80        61
   macro avg       0.80      0.80      0.80        61
weighted avg       0.80      0.80      0.80        61

ROC-AUC: 0.9047619047619048

🔹 Tuning Decision Tree...
Best Params for Decision Tree: {'max_depth': 4, 'min_samples_split': 10}
Best CV Accuracy: 0.8256

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.85      0.79        33
           1       0.78      0.64      0.71        28

    accuracy                           0.75        61
   macro avg       0.76      0.75      0.75        61
weighted avg       0.76      0.75      0.75        61

ROC-AUC: 0.78679653

In [12]:
scores = {}
for name, model in best_models.items():
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        scores[name] = roc_auc_score(y_test, y_proba)

best_model_name = max(scores, key=scores.get)
final_model = best_models[best_model_name]

In [14]:
joblib.dump(final_model, "../models/best_model.pkl")


['../models/best_model.pkl']